#### Creating a raster data cube for zonal statistics

In [ ]:
index_dir = Path(r"C:\Users\JULIA-PC\OneDrive - Universität Salzburg\MorphEO\data\OBIA_LS_lake_outlines")

# ── Find all index rasters ──
index_names = ["nddi", "ndvi", "ndsi", "ndwi", "brightness"]
pattern = re.compile(r"^(" + "|".join(index_names) + r")_(\d{4})\.tif$", re.IGNORECASE)

# Collect all files into {index: {year: path}}
index_files = {}
for f in index_dir.glob("*.tif"):
    m = pattern.match(f.name)
    if m:
        idx  = m.group(1).lower()
        year = int(m.group(2))
        index_files.setdefault(idx, {})[year] = f

print("Found files:")
for idx, years in sorted(index_files.items()):
    print(f"  {idx}: {sorted(years.keys())}")

In [ ]:
# ── Load and align all rasters ──
# Use the first raster as the reference grid for reprojection

# Find reference CRS and transform from the first available file
ref_path = next(iter(next(iter(index_files.values())).values()))
ref = rioxarray.open_rasterio(ref_path, masked=True).squeeze("band", drop=True)
ref_crs = ref.rio.crs

# Collect all years across all indices
all_years = sorted(set(yr for years in index_files.values() for yr in years))
print(f"All years: {all_years}")
print(f"Reference CRS: {ref_crs}")

In [ ]:
# ── Build the raster cube: (index × year × x × y) ──
# For missing (index, year) combinations, fill with NaN

def load_raster(path, ref):
    """Load a raster, reproject to reference grid if needed."""
    da = rioxarray.open_rasterio(path, masked=True).squeeze("band", drop=True)
    if da.rio.crs != ref.rio.crs or da.shape != ref.shape:
        da = da.rio.reproject_match(ref)
    return da

index_arrays = []
for idx in index_names:
    year_arrays = []
    for yr in all_years:
        if yr in index_files.get(idx, {}):
            da = load_raster(index_files[idx][yr], ref)
        else:
            # Missing year — fill with NaN at reference grid shape
            da = xr.full_like(ref, fill_value=np.nan)
        da = da.expand_dims(year=[yr])
        year_arrays.append(da)
    
    # Stack years for this index
    idx_stack = xr.concat(year_arrays, dim="year")
    idx_stack = idx_stack.expand_dims(index=[idx])
    index_arrays.append(idx_stack)

# Concatenate all indices
raster_cube = xr.concat(index_arrays, dim="index")
raster_cube.name = "spectral_index"

raster_cube

In [ ]:
raster_cube.plot(row='year', col='index')

In [ ]:
# Flatten outlines to a GeoSeries with a MultiIndex
geoseries = (
    outlines_ds["geometry"]
    .to_series()                        # pandas Series with (id, year) MultiIndex
    .dropna()                           # remove missing lake/year combos
    .pipe(lambda s: gpd.GeoSeries(s, crs=all_lakes.crs))
)

aggregated = raster_cube.xvec.zonal_stats(
    geoseries,
    x_coords="x",
    y_coords="y",
)

In [ ]:
aggregated